# RouteRanker Module 5: Clustering and Evaluation

This notebook loads the Module 4 prepared route features, runs K-Means and DBSCAN, displays evaluation tables, and shows the generated visualizations.

## Open in VS Code

1. Open the RouteRanker project folder in VS Code.
2. Open this notebook and choose a Python kernel.
3. If needed, install the notebook dependencies from the project terminal with `python -m pip install -r requirements-notebook.txt`.
4. Run the cells from top to bottom. The model run writes outputs under `models/`, `results/`, `reports/`, and `visualizations/`.

The clustering describes similarity between scheduled route features. It does not measure passenger demand or route quality.


In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import SVG, display

# Find this project whether VS Code starts the notebook from the project root or notebooks/.
search_locations = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (
        folder for folder in search_locations
        if (folder / "data" / "processed" / "routes_ml_ready.csv").is_file()
        and (
            (folder / "module5_clustering.py").is_file()
            or (folder / "src" / "route_ranker" / "module5_clustering.py").is_file()
        )
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the RouteRanker project. Open the project folder in VS Code, "
        "or set PROJECT_ROOT manually to the folder containing data/processed/routes_ml_ready.csv."
    )

DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "routes_ml_ready.csv"
print(f"Project folder: {PROJECT_ROOT}")
print(f"Module 4 dataset: {DATASET_PATH}")


Project folder: C:\Users\dhana\OneDrive\Desktop\ROUT
Module 4 dataset: C:\Users\dhana\OneDrive\Desktop\ROUT\data\processed\routes_ml_ready.csv


In [ ]:
# Import the standalone Desktop script or the packaged src/ module.
if (PROJECT_ROOT / "module5_clustering.py").is_file():
    sys.path.insert(0, str(PROJECT_ROOT))
    import module5_clustering as module5
else:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
    from route_ranker import module5_clustering as module5

routes, feature_columns = module5.read_features(PROJECT_ROOT, DATASET_PATH)
print(f"Loaded {len(routes):,} routes and {len(feature_columns)} model features.")
display(routes[[c for c in ["city", "route_id", "route_name", "split"] if c in routes.columns] + feature_columns[:5]].head())


## Run the Module 5 models

This cell performs the full clustering workflow. It may take a short while to complete.

In [ ]:
summary = module5.run(PROJECT_ROOT, dataset_path=DATASET_PATH, seed=42)
display(pd.DataFrame([summary]))


## Evaluation results

The DBSCAN table distinguishes parameter-sweep sample scores from the selected fit on all training rows. Classification scores are only available after supplying genuine route labels.

In [ ]:
results_dir = PROJECT_ROOT / "results"
for filename in ["kmeans_metrics.csv", "dbscan_metrics.csv", "classification_metrics.csv", "algorithm_comparison.csv"]:
    csv_path = results_dir / filename
    print(f"\n{filename}")
    if csv_path.exists():
        display(pd.read_csv(csv_path))
    else:
        print("Not generated yet.")


## Cluster profiles and route review shortlist

These tables summarize the generated groups and the schedule-only human review list. The review score is not a ridership or demand score.

In [ ]:
for filename in ["cluster_profiles.csv", "route_review_candidates.csv"]:
    csv_path = results_dir / filename
    print(f"\n{filename}")
    if csv_path.exists():
        display(pd.read_csv(csv_path).head(20))
    else:
        print("Not generated yet.")


## Visualizations

In [ ]:
visualization_dir = PROJECT_ROOT / "visualizations"
for filename in [
    "kmeans_elbow.svg",
    "kmeans_clusters_pca.svg",
    "dbscan_clusters_pca.svg",
    "dbscan_k_distance.svg",
]:
    image_path = visualization_dir / filename
    print(filename)
    if image_path.exists():
        display(SVG(filename=str(image_path)))
    else:
        print("Run the model cell above to generate this visualization.")


## Interpretation and limitations

- K-Means assigns every route to one of the selected K groups; the elbow and internal metrics help compare candidate K values.
- DBSCAN groups dense neighborhoods and marks sparse rows as noise. Its result depends on feature scaling and its `eps` and `min_samples` settings.
- These are unsupervised clusters. They are not validated route categories, demand estimates, or service-quality scores.
- Use passenger counts, operating constraints, transfer patterns, and community access before making route changes.
